# Vlasov-Ampere block-encoding Circuit experiment

Execute the complete original grid and source demonstration QSVT pair.

## The migrated block encoding runs on FlagQuantum

The displayed block factory constructs the 17-bit finite-difference encoding
from velocity, force, current and boundary helpers. Despite the historical
folder name, execution uses FlagQuantum Circuit and its native CPU simulator,
without a Qiskit execution dependency. This remains explicit Circuit kernel
construction, not a fully synthesized high-level qfunc workflow.

Inspect the unnormalized selector-zero projection against the independent
128-by-128 matrix and check work leakage separately. Velocity and position
have three bits each; boundary normalization and bit order affect meaning.
Changing force or boundaries requires updating both models. The optional
18-bit QSVT step is construction evidence, not a full inverse-solver result.


Use the installed FlagQuantum runtime.

In [ ]:
import flagquantum as fq
import torch

from examples.synthesis.applications.plasma.vlasov_ampere.vlasov_ampere_qiskit.overview.quantum import (
    ADVECTION_NORM,
    Builder,
    CURRENT_NORM,
    DTYPE,
    FORCE,
    FORCE_NORM,
    NORMALIZATION,
    OFF_DIAGONAL_NORM,
    VELOCITY,
    assign,
    derivative,
    prepare,
)


def block_preview() -> fq.Circuit:
    """Retain seven data bits and all ten original block/scratch bits."""
    circuit = fq.Circuit(17, dtype=DTYPE)
    builder = Builder(circuit)
    amplitudes = torch.sqrt(
        torch.tensor([ADVECTION_NORM, OFF_DIAGONAL_NORM, 0.8, 0], dtype=torch.float64)
        / NORMALIZATION
    )
    builder.emit(prepare(amplitudes), (7, 8))
    branch = ((7, 0), (8, 0))
    derivative(builder, branch)
    assign(builder, VELOCITY / 4, (0, 1, 2), 12, branch)
    builder.flip(12, branch)
    # Source's two scratch-computed zeta predicates are uncomputed exactly.
    for x_value, negative in ((7, 1), (0, 0)):
        predicate = branch + tuple(
            (wire, (x_value >> i) & 1) for i, wire in enumerate((3, 4, 5))
        )
        builder.flip(15, predicate)
        builder.flip(16, branch + ((2, negative),))
        builder.flip(13, branch + ((15, 1), (16, 1)))
        builder.flip(16, branch + ((2, negative),))
        builder.flip(15, predicate)
    builder.flip(14, branch + ((6, 1),))
    branch = ((7, 1), (8, 0))
    builder.flip(6, branch)
    force = branch + ((6, 0),)
    current = branch + ((6, 1),)
    builder.flip(9, force + ((0, 0), (1, 0), (2, 0)))
    builder.flip(9, force)
    builder.emit(prepare(-FORCE / FORCE_NORM), (0, 1, 2), force)
    builder.emit(prepare(VELOCITY / VELOCITY.norm()).mH, (0, 1, 2), current)
    builder.flip(9, current + ((0, 0), (1, 0), (2, 0)))
    builder.flip(9, current)
    ratio = FORCE_NORM / CURRENT_NORM
    equalization = torch.tensor([ratio, 1.0], dtype=torch.float64)
    equalization /= equalization.max()
    assign(builder, equalization, (6,), 12, branch)
    builder.flip(12, branch)
    # P(pi) RZ(-pi) equals iI exactly, with the selector branch retained.
    builder.emit(1j * torch.eye(2, dtype=DTYPE), (9,), ((7, 0), (8, 1)))
    final = amplitudes.clone()
    final[0] *= -1
    builder.emit(prepare(final).mH, (7, 8))
    return circuit


from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.plasma.vlasov_ampere.vlasov_ampere_qiskit.overview import (
    main,
    physics,
    plots,
    quantum,
    workflow,
)

print("Execution uses the existing native CPU runtime")


Load native Circuit composition and fresh physics helpers.

In [ ]:
build_encoding = quantum.block_encoding


Use an exact modular shift with the same complete four-bit permutation as the source Draper adder.

In [ ]:
minus_one = quantum.shift(4, -1)
plus_one = quantum.shift(4, 1)


Use explicit uniformly controlled real rotations for amplitude assignment.

In [ ]:
assign_amplitudes = quantum.assign


Keep the original 8 by 8 phase-space grid, temperature, source and domain.

In [ ]:
assert quantum.VELOCITY.numel() == 8
spatial_points = velocity_points = 8


Freshly normalize the original boundary coefficients.

In [ ]:
boundary_norm = quantum.BC_NORM


Compose every derivative, boundary, signed-velocity, zeta and E selection branch.

In [ ]:
derivative_builder = quantum.derivative


Retain force, current and unequal branch normalization.

In [ ]:
force_norm = quantum.FORCE_NORM
current_norm = quantum.CURRENT_NORM


Compute original term scale factors directly.

In [ ]:
print(
    f"Advection normalization: {quantum.ADVECTION_NORM}; off-diagonal normalization: {quantum.OFF_DIAGONAL_NORM}"
)


Preserve the controlled iI branch and all relative global phases.

In [ ]:
normalization = quantum.NORMALIZATION


Build the original seven-data-bit, ten-block-bit encoding.

In [ ]:
encoding = block_preview()
assert encoding.n_qubits == 17


Retain the complete normalization sum.

In [ ]:
print(f"Block normalization: {normalization}")


Construct the fresh classical advection and off-diagonal operator.

In [ ]:
classical_matrix = physics.matrix()
assert classical_matrix.shape == (128, 128)


Keep both E sectors and all phase-space indices.

In [ ]:
assert classical_matrix.numel() == 16384


Generate a fresh 128-amplitude random input with an explicit local seed.

In [ ]:
initial = physics.initial_state(17)


Run the complete native encoding and report its actual static basis.

In [ ]:
result = workflow.run(17)


Execute the entire 17-bit state before selecting the zero block.

In [ ]:
assert result.block_resources.n_qubits == 17


Read the actual unnormalized block-zero projection.

In [ ]:
projected = result.projected


Apply the independent classical matrix to the identical input.

In [ ]:
expected = result.expected


Compare every original data-basis amplitude.

In [ ]:
figure_directory = Path("vlasov-circuit-figures")
for filename in plots.save(result, figure_directory):
    display(Image(filename=str(figure_directory / filename)))


Check the source complex-vector L2 threshold and clean auxiliary bits.

In [ ]:
assert result.l2_error < 1e-10
assert result.work_leak < 1e-24


Retain both eight-bit block-zero projectors and the separate QSVT auxiliary.

In [ ]:
projector_bits = tuple(range(7, 15))


Build the source single QSVT demonstration pair with angles 0.1 and 0.2.

In [ ]:
qsvt_circuit = quantum.qsvt_step(0.1, 0.2)


Keep all original data, block and auxiliary bits.

In [ ]:
assert qsvt_circuit.n_qubits == 18


Report actual native static resources and persist the observed projection.

In [ ]:
main.report(result)
main.write(result, Path("vlasov-circuit-results.json"))
